# Phase 0 — Environment Smoke Test and Information Boundary Audit

This notebook verifies that:
1. Crafter observation shape is exactly `[64, 64, 3]` and dtype is `uint8`.
2. The agent selects anonymous tokens in `0..16`, translated by `OpaqueActionMapping`.
3. No `info` dictionary, achievement string, or internal game state leaks to the learner.
4. `EvaluatorTap` records metrics strictly on an isolated one-way side-channel.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
from src.env.crafter_adapter import CrafterAdapter
from src.env.evaluator_tap import EvaluatorTap
from src.env.learner_observation_filter import LearnerObservationFilter

# 1. Initialize environment adapter with EvaluatorTap
tap = EvaluatorTap()
adapter = CrafterAdapter(seed=42, evaluator_tap=tap)
obs = adapter.reset()

print(f"Observation shape: {obs.shape}, dtype: {obs.dtype}")
assert obs.shape == (64, 64, 3)
assert obs.dtype == np.uint8

# 2. Step 10 actions and verify information filter
for t in range(10):
    token = t % 17
    next_obs, reward, terminated, truncated = adapter.step(token)
    assert len((next_obs, reward, terminated, truncated)) == 4
    LearnerObservationFilter.inspect_against_leakage((next_obs, reward, terminated, truncated))

print("Smoke test passed! 0 leaks detected; contract 100% compliant.")
adapter.close()